In [1]:
import pandas as pd
import random
import json
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from huggingface_hub import InferenceClient
from kaggle_secrets import UserSecretsClient


In [4]:
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HUGGING_FACEE_API_KEY")

client = InferenceClient(api_key=HF_TOKEN)

modelo = "meta-llama/Llama-3.1-8B-Instruct"

temperature=0.3
top_p=0.9
max_tokens=700

In [5]:
dados_exemplo = {
    "temperatura_motor": random.randint(60, 120),
    "nivel_combustivel": random.randint(5, 100),
    "pressao_cabine": round(random.uniform(0.6, 1.2), 2),
    "oxigenio": random.randint(50, 100),
    "bateria": random.randint(10, 100),
    "radiacao": round(random.uniform(0.1, 2.5), 2),
    "velocidade": random.randint(24000, 29000),
    "distancia_da_terra_km": random.randint(300000, 500000),
    "sinal_comunicacao": random.randint(20, 100)
}

campos_obrigatorios = [
    "temperatura_motor",
    "nivel_combustivel",
    "pressao_cabine",
    "oxigenio",
    "bateria",
    "radiacao",
    "velocidade",
    "distancia_da_terra_km",
    "sinal_comunicacao"
]

entrada_dados_missao = widgets.Textarea(
    value=json.dumps(dados_exemplo, indent=4, ensure_ascii=False),
    placeholder="Cole ou digite aqui os dados da missão...",
    description="Dados:",
    layout=widgets.Layout(width="100%", height="300px")
)

botao_carregar_dados = widgets.Button(
    description="Carregar dados da missão",
    button_style="success",
    icon="check",
    layout=widgets.Layout(width="max-content")
)

saida_dados_missao = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #e2e8f0",
        padding="10px",
        margin_top="10px"
    )
)

def carregar_dados_missao(_=None):
    global dados_missao, df

    with saida_dados_missao:
        clear_output()

        try:
            dados = json.loads(entrada_dados_missao.value)

            campos_faltando = [campo for campo in campos_obrigatorios if campo not in dados]
            if campos_faltando:
                raise ValueError(f"Campos obrigatórios ausentes: {campos_faltando}")

            dados_missao = {
                "temperatura_motor": int(dados["temperatura_motor"]),
                "nivel_combustivel": int(dados["nivel_combustivel"]),
                "pressao_cabine": float(dados["pressao_cabine"]),
                "oxigenio": int(dados["oxigenio"]),
                "bateria": int(dados["bateria"]),
                "radiacao": float(dados["radiacao"]),
                "velocidade": int(dados["velocidade"]),
                "distancia_da_terra_km": int(dados["distancia_da_terra_km"]),
                "sinal_comunicacao": int(dados["sinal_comunicacao"])
            }

            df = pd.DataFrame([dados_missao])

            print("Dados da missão carregados com sucesso!")
            print("Confira abaixo os valores que serão analisados pela IA:")
            display(df)

        except json.JSONDecodeError:
            print("Erro: os dados precisam estar em formato válido.")
            print("Dica: mantenha aspas nos nomes dos campos e use vírgulas entre os valores.")
        except Exception as erro:
            print(f"Erro ao carregar os dados: {erro}")

botao_carregar_dados.on_click(carregar_dados_missao)

display(HTML("<h3>Insira os dados da missão</h3>"))
display(HTML("<p>Altere os valores abaixo e clique em <b>Carregar dados da missão</b>. Depois, execute a célula final para gerar a resposta da IA.</p>"))
display(entrada_dados_missao, botao_carregar_dados, saida_dados_missao)

carregar_dados_missao()


Textarea(value='{\n    "temperatura_motor": 120,\n    "nivel_combustivel": 39,\n    "pressao_cabine": 0.67,\n …

Button(button_style='success', description='Carregar dados da missão', icon='check', layout=Layout(width='max-…

Output(layout=Layout(border_bottom='1px solid #e2e8f0', border_left='1px solid #e2e8f0', border_right='1px sol…

In [7]:
def classificar_riscos(dados):
    riscos = []

    if dados["temperatura_motor"] > 100:
        riscos.append("Temperatura do motor acima do limite seguro.")

    if dados["nivel_combustivel"] < 20:
        riscos.append("Nível de combustível crítico.")

    if dados["pressao_cabine"] < 0.8:
        riscos.append("Pressão da cabine abaixo do recomendado.")

    if dados["oxigenio"] < 60:
        riscos.append("Oxigênio em nível preocupante.")

    if dados["bateria"] < 25:
        riscos.append("Bateria em nível crítico.")

    if dados["radiacao"] > 1.8:
        riscos.append("Radiação elevada detectada.")

    if dados["sinal_comunicacao"] < 40:
        riscos.append("Sinal de comunicação instável.")

    if not riscos:
        riscos.append("Nenhum risco crítico detectado no momento.")

    return riscos

In [12]:
def criar_prompt(dados):
    riscos = classificar_riscos(dados)

    prompt = f"""
Você é uma IA especialista em monitoramento de missões espaciais.
Seu papel é atuar como um sistema de apoio à decisão para uma central de controle.

Analise cuidadosamente os dados abaixo e responda com raciocínio técnico, claro e objetivo.

DADOS DA MISSÃO:

- Temperatura do motor: {dados["temperatura_motor"]} °C
- Nível de combustível: {dados["nivel_combustivel"]}%
- Pressão da cabine: {dados["pressao_cabine"]} atm
- Oxigênio disponível: {dados["oxigenio"]}%
- Bateria: {dados["bateria"]}%
- Radiação externa: {dados["radiacao"]} mSv/h
- Velocidade da nave: {dados["velocidade"]} km/h
- Distância da Terra: {dados["distancia_da_terra_km"]} km
- Sinal de comunicação: {dados["sinal_comunicacao"]}%

RISCOS IDENTIFICADOS PELO SISTEMA:
{riscos}

INSTRUÇÕES PARA SUA RESPOSTA:

1. Faça uma análise geral do status da missão.
2. Identifique possíveis falhas ou riscos futuros.
3. Explique quais dados indicam perigo.
4. Recomende ações automáticas ou humanas.
5. Classifique o status final da missão como:
   - Estável
   - Atenção
   - Crítico

Responda em português, com tom técnico, direto e organizado.
Não invente dados que não foram fornecidos.
"""
    return prompt

In [11]:
def gerar_resposta_ia(dados):
    prompt = criar_prompt(dados)

    resposta = client.chat_completion(
        model=modelo,
        messages=[
            {
                "role": "system",
                "content": "Você é uma IA técnica, precisa, cautelosa e especializada em segurança aeroespacial."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.3,
        max_tokens=700,
        top_p=0.9
    )

    return resposta.choices[0].message.content

botao_gerar_relatorio = widgets.Button(
    description="Gerar relatório inteligente",
    button_style="primary",
    icon="rocket",
    layout=widgets.Layout(width="max-content")
)

saida_relatorio = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #e2e8f0",
        padding="10px",
        margin_top="10px"
    )
)

def gerar_relatorio(_=None):
    with saida_relatorio:
        clear_output()

        print(" RISCOS CLASSIFICADOS PELO SISTEMA ")
        for risco in classificar_riscos(dados_missao):
            print(f"- {risco}")

        print("\n RELATÓRIO INTELIGENTE DA MISSÃO \n")

        resposta = gerar_resposta_ia(dados_missao)
        print(resposta)

botao_gerar_relatorio.on_click(gerar_relatorio)

display(HTML("<h3>Resposta da IA</h3>"))
display(HTML("<p>Clique no botão abaixo para a IA analisar os dados carregados e responder mantendo o formato técnico solicitado.</p>"))
display(botao_gerar_relatorio, saida_relatorio)


Button(button_style='primary', description='Gerar relatório inteligente', icon='rocket', layout=Layout(width='…

Output(layout=Layout(border_bottom='1px solid #e2e8f0', border_left='1px solid #e2e8f0', border_right='1px sol…